# Train and evaluate TeraiNet

1. Set variables and parameters
2. Prepare train, val and test datasets
3. Prepare model
4. Train model
5. Evaluate model
6. Log run to W&B

## Setup

### Change working directory

In [ ]:
%cd ../../

### Clone TeraiNet repo

In [ ]:
!git clone -b chore/refactoring https://github.com/alexvmt/terainet.git

### Imports

Follow [mewc-flow](https://github.com/zaandahl/mewc-flow/blob/main/requirements.txt) for the key package versions

In [ ]:
!pip install keras==3.3.3 kimm==0.2.5 tensorflow==2.16.1

In [ ]:
import logging
import random
import shutil
import sys
import time
from pathlib import Path

import keras
import kimm
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import wandb
from kaggle_secrets import UserSecretsClient
from keras import callbacks, losses, optimizers, saving
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

project_dir = "terainet"
sys.path.insert(0, f"{project_dir}/src")

from terainet import (
    create_class_list_yaml_file,
    filter_single_snippet_images,
    load_config,
    sample_images,
)

logging.basicConfig(
    level=logging.INFO,
    stream=sys.stdout,
    force=True,
)

logger = logging.getLogger(__name__)

### Set variables and parameters

In [ ]:
# load config
config_path = Path(project_dir) / "config.yaml"
config = load_config(str(config_path))

# set num classes and class names
num_classes = config["classes"]["num_classes"]
class_names = [key for key in config["classes"] if key != "num_classes"]

# set seed
seed = 42

# filter images
filter_images = True

# set n train, val and test images
n_train_images = 2000
n_val_images = 250
n_test_images = 250

# define paths to train, val and test images
images_input_dir = Path(config["training"]["images_input_dir"])
images_filtered_dir = Path(config["training"]["images_filtered_dir"])
images_filtered_dir.mkdir(parents=True, exist_ok=True)
images_sampled_dir = Path(config["training"]["images_sampled_dir"])
images_sampled_dir.mkdir(parents=True, exist_ok=True)

train_dir = images_input_dir / config["training"]["train_dir"]
train_filtered_dir = images_filtered_dir / config["training"]["train_filtered_dir"]
train_sampled_dir = images_sampled_dir / config["training"]["train_sampled_dir"]

val_dir = images_input_dir / config["training"]["val_dir"]
val_filtered_dir = images_filtered_dir / config["training"]["val_filtered_dir"]
val_sampled_dir = images_sampled_dir / config["training"]["val_sampled_dir"]

test_dir = images_input_dir / config["training"]["test_dir"]
test_filtered_dir = images_filtered_dir / config["training"]["test_filtered_dir"]
test_sampled_dir = images_sampled_dir / config["training"]["test_sampled_dir"]

test2_dir = images_input_dir / config["training"]["test2_dir"]

# model
model_constructor = "EfficientNetV2M"
model_path = config["training"]["model_path"]
classification_report_path = config["training"]["classification_report_path"]
confusion_matrix_path = config["training"]["confusion_matrix_path"]

# set image size
img_size = 224

# set batch size
batch_size = 8

# augmentations
apply_augmentations = False

# decide whether to log run or not
wb_project = config["training"]["wb_project"]
use_wandb_logging = True

In [ ]:
# create class list yaml file which is needed later for deployment
class_list_yaml_path = config["training"]["class_list_yaml_path"]
create_class_list_yaml_file(num_classes, class_names, class_list_yaml_path)

In [ ]:
# log in to w&b using api key
if use_wandb_logging:
    user_secrets = UserSecretsClient()
    key = user_secrets.get_secret("wandb")
    !wandb login $key

## Prepare train, val and test datasets

### Filter to keep only single-snippet images

In [ ]:
if filter_images:
    exclude_classes = ["class_1"]
    filter_stats = filter_single_snippet_images(
        images_input_dir, images_filtered_dir, exclude_classes=exclude_classes
    )

    # View detailed statistics
    for subset, stats_dict in filter_stats.items():
        print(f"\n{subset.upper()}:")
        print(
            f"  Total: {stats_dict['original_total']} → {stats_dict['filtered_total']} "
            f"({stats_dict['removed_total']} removed)"
        )
        for class_name, class_stats in stats_dict["per_class"].items():
            removed_pct = (
                (class_stats["removed"] / class_stats["original"] * 100)
                if class_stats["original"] > 0
                else 0
            )
            print(
                f"  {class_name}: {class_stats['original']} → {class_stats['filtered']} "
                f"({removed_pct:.1f}% removed)"
            )

### Sample images

In [ ]:
# create new directory with sampled train images
if filter_images:
    sample_images(train_filtered_dir, train_sampled_dir, n_train_images)
else:
    sample_images(train_dir, train_sampled_dir, n_train_images)

In [ ]:
# create new directory with sampled val images
if filter_images:
    sample_images(val_filtered_dir, val_sampled_dir, n_val_images)
else:
    sample_images(val_dir, val_sampled_dir, n_val_images)

In [ ]:
# TODO: Move this to sampling/downloading

# Quick fix!
# The images from the Amur tiger re-identification challenge
# come already split into train and test.
# We therefore need to sample a separate val set
# from the existing train set ourselves.
# Ideally this should be done on raw images that haven't yet
# been processed by MegaDetector and subsequently cropped.
# For now, this is done on cropped images here, ensuring that
# the sampled val images don't stem from the same raw images
# as those images sampled for train.

# create target dir
class_dir = "class_1"
target_dir = val_sampled_dir / class_dir
target_dir.mkdir(parents=True, exist_ok=True)

# get excluded prefixes
exclude_dir = train_sampled_dir / class_dir
exclude_prefixes = {f.name.split("-")[0] for f in exclude_dir.iterdir() if f.is_file()}

# get eligible files
source_dir = train_dir / class_dir
eligible_files = [
    f.name for f in source_dir.iterdir() if f.name.split("-")[0] not in exclude_prefixes
]

# reproducible sampling
random.seed(seed)
sampled_files = random.sample(eligible_files, n_val_images)

# copy files
for f in sampled_files:
    shutil.copy2(source_dir / f, target_dir / f)

In [ ]:
# create new directory with sampled test images
if filter_images:
    sample_images(test_filtered_dir, test_sampled_dir, n_test_images)
else:
    sample_images(test_dir, test_sampled_dir, n_test_images)

### Create datasets

In [ ]:
# create train dataset
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_sampled_dir,
    label_mode="categorical",
    shuffle=True,
    seed=seed,
)

# create validation dataset
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    val_sampled_dir,
    label_mode="categorical",
    shuffle=False,
)

# create test dataset
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_sampled_dir,
    label_mode="categorical",
    shuffle=False,
)

# create test2 dataset
test2_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test2_dir,
    label_mode="categorical",
    shuffle=False,
)

# we need to unbatch because there's somehow an unwanted additional dimension
train_ds = train_ds.unbatch()
val_ds = val_ds.unbatch()
test_ds = test_ds.unbatch()
test2_ds = test2_ds.unbatch()

logger.info(f"Number of train samples: {train_ds.cardinality()}")
logger.info(f"Number of val samples: {val_ds.cardinality()}")
logger.info(f"Number of test samples: {test_ds.cardinality()}")

In [ ]:
# check dimensions
logger.info(f"Train element spec: {train_ds.element_spec}")
logger.info(f"Val element spec: {val_ds.element_spec}")
logger.info(f"Test element spec: {test_ds.element_spec}")

### Resize (and augment) images

In [ ]:
# resize images and setup datasets with tf.data
resize_fn = keras.layers.Resizing(img_size, img_size)

# define augmentations
data_augmentations = keras.Sequential(
    [
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.05),
        keras.layers.RandomZoom(0.1),
        keras.layers.RandomContrast(0.1),
        keras.layers.Lambda(lambda x: tf.image.random_brightness(x, max_delta=0.1)),
        keras.layers.Lambda(lambda x: tf.image.random_saturation(x, lower=0.9, upper=1.1)),
        keras.layers.Lambda(lambda x: tf.image.random_hue(x, max_delta=0.02)),
    ]
)

# apply resizing + augmentations to training data
if apply_augmentations:
    train_ds = train_ds.map(lambda x, y: (data_augmentations(resize_fn(x), training=True), y))
else:
    train_ds = train_ds.map(lambda x, y: (resize_fn(x), y))

# only resize validation and test data
val_ds = val_ds.map(lambda x, y: (resize_fn(x), y))
test_ds = test_ds.map(lambda x, y: (resize_fn(x), y))
test2_ds = test2_ds.map(lambda x, y: (resize_fn(x), y))

# cache train_ds only if augmentations are not used
if apply_augmentations:
    train_ds = train_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
else:
    train_ds = train_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE).cache()
val_ds = val_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE).cache()
test_ds = test_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE).cache()
test2_ds = test2_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE).cache()

## Prepare model

In [ ]:
# create base model
if model_constructor == "EfficientNetV2B0":
    base_model = kimm.models.EfficientNetV2B0(
        input_shape=(img_size, img_size, 3),
        include_preprocessing=True,
        include_top=False,
    )
    model_params = "5M"
    frozen_file_size = "26MB"
elif model_constructor == "EfficientNetV2B2":
    base_model = kimm.models.EfficientNetV2B2(
        input_shape=(img_size, img_size, 3),
        include_preprocessing=True,
        include_top=False,
    )
    model_params = "9M"
    frozen_file_size = "37MB"
elif model_constructor == "EfficientNetV2S":
    base_model = kimm.models.EfficientNetV2S(
        input_shape=(img_size, img_size, 3),
        include_preprocessing=True,
        include_top=False,
    )
    model_params = "21M"
    frozen_file_size = "84MB"
elif model_constructor == "EfficientNetV2M":
    base_model = kimm.models.EfficientNetV2M(
        input_shape=(img_size, img_size, 3),
        include_preprocessing=True,
        include_top=False,
    )
    model_params = "54M"
    frozen_file_size = "216MB"
elif model_constructor == "EfficientNetV2L":
    base_model = kimm.models.EfficientNetV2L(
        input_shape=(img_size, img_size, 3),
        include_preprocessing=True,
        include_top=False,
    )
    model_params = "119M"
    frozen_file_size = "475MB"
elif model_constructor == "EfficientNetV2XL":
    base_model = kimm.models.EfficientNetV2XL(
        input_shape=(img_size, img_size, 3),
        include_preprocessing=True,
        include_top=False,
    )
    model_params = "208M"
    frozen_file_size = "835MB"
else:
    raise Exception("Please select a valid model constructor.")

# freeze base model
base_model.trainable = False

# create new model on top
inputs = keras.Input(shape=(img_size, img_size, 3))
x = inputs

# The base model contains batchnorm layers. We want to keep them in inference mode
# when we unfreeze the base model for fine-tuning, so we make sure that the
# base_model is running in inference mode here.
x = base_model(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.2)(x)  # regularize with dropout
outputs = keras.layers.Dense(num_classes, activation="softmax")(x)
model = keras.Model(inputs, outputs)

model.summary(show_trainable=True)

## Train model

Follow [mewc-train](https://github.com/zaandahl/mewc-train) for the training parameters

In [ ]:
df_size = int(n_train_images * num_classes)
epochs = 50
lr_init = 1e-4
min_lr_frac = 1 / 5  # default minimum learning rate fraction of initial learning rate
steps_per_epoch = df_size // batch_size
total_steps = (
    epochs * steps_per_epoch
)  # total number of steps for monotonic exponential decay across all epochs
lr = optimizers.schedules.ExponentialDecay(
    initial_learning_rate=lr_init, decay_steps=total_steps, decay_rate=min_lr_frac, staircase=False
)
amsgrad = True
weight_decay = 1e-4
optimizer = optimizers.AdamW(learning_rate=lr, amsgrad=amsgrad, weight_decay=weight_decay)

# EDIT: switching to non-focal loss because dataset is balanced
# EDIT: use label smoothing to prevent overconfident predictions
# ensure that loss can be calculated across gpus
if num_classes == 2:
    loss_f = losses.BinaryCrossentropy(label_smoothing=0.1)  # use for binary classification tasks
    act_f = "sigmoid"  # use for binary classification tasks
else:
    loss_f = losses.CategoricalCrossentropy(
        label_smoothing=0.1
    )  # use for unbalanced multi-class tasks (typical for wildlife datasets)
    act_f = "softmax"  # use for multi-class classification tasks

metrics = ["accuracy"]

callbacks = [
    callbacks.EarlyStopping(
        monitor="loss", mode="min", min_delta=0.001, patience=5, restore_best_weights=True
    )
]

In [ ]:
# compile model
model.compile(
    optimizer=optimizer,
    loss=loss_f,
    metrics=metrics,
)

In [ ]:
# train model
start_time = time.time()
history = model.fit(train_ds, epochs=epochs, callbacks=callbacks, validation_data=val_ds)
end_time = time.time()
training_time_mins = round((end_time - start_time) / 60, 2)
logger.info(f"Training time mins: {training_time_mins}")

In [ ]:
# visualize train and val loss
train_loss = history.history["loss"]
val_loss = history.history.get("val_loss")

plt.figure(figsize=(10, 6))
plt.plot(train_loss, label="Training loss")
plt.plot(val_loss, label="Validation loss")

plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and validation loss")
plt.legend()
plt.grid()
plt.gca().xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.show()

In [ ]:
saving.save_model(model, model_path, include_optimizer=False)

## Evaluate model

### Get predictions on test dataset

In [ ]:
test_accuracy = model.evaluate(test_ds)

In [ ]:
test2_accuracy = model.evaluate(test2_ds)

In [ ]:
true_labels = []
predicted_labels = []

for images, labels in test_ds:
    # append true labels based on their format
    if len(labels.shape) > 1:  # if one-hot encoded
        true_labels.append(np.argmax(labels.numpy(), axis=1))
    else:  # if integer labels
        true_labels.append(labels.numpy())

    # predict labels
    predictions = model.predict(images)
    predicted_labels.append(np.argmax(predictions, axis=1))

# combine all batches into single arrays
true_labels = np.concatenate(true_labels)
predicted_labels = np.concatenate(predicted_labels)

### Get performance metrics, classification report and confusion matrix

In [ ]:
precision = precision_score(true_labels, predicted_labels, average="macro")
recall = recall_score(true_labels, predicted_labels, average="macro")
f1 = f1_score(true_labels, predicted_labels, average="macro")
print(
    f"Test accuracy: {test_accuracy[1]:.4f}\nTest2 accuracy: {test2_accuracy[1]:.4f}\nTest precision: {precision:.4f}\nTest recall: {recall:.4f}\nTest f1-score: {f1:.4f}"
)

In [ ]:
cr_dict = classification_report(
    true_labels, predicted_labels, target_names=class_names, output_dict=True
)

# remove scalar 'accuracy'
cr_dict.pop("accuracy", None)

# add micro avg manually
micro_avg = {
    "precision": precision_score(true_labels, predicted_labels, average="micro"),
    "recall": recall_score(true_labels, predicted_labels, average="micro"),
    "f1-score": f1_score(true_labels, predicted_labels, average="micro"),
    "support": len(true_labels),
}
cr_dict["micro avg"] = micro_avg

# reorder: class entries + micro avg + macro avg + weighted avg
ordered_keys = class_names + ["micro avg", "macro avg", "weighted avg"]

# rebuild dict with correct order
ordered_cr_dict = {k: cr_dict[k] for k in ordered_keys if k in cr_dict}

cr_df = pd.DataFrame(ordered_cr_dict).transpose()
cr_df.to_csv(classification_report_path)
print("Classification report:\n", cr_df)

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)
print("Confusion matrix:\n", cm)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion matrix")
plt.savefig(confusion_matrix_path, dpi=300, bbox_inches="tight")
plt.show()

## Log run to W&B

In [ ]:
if use_wandb_logging:
    run = wandb.init(
        project=wb_project,
        config={
            "model_constructor": model_constructor,
            "model_params": model_params,
            "frozen_file_size": frozen_file_size,
            "img_size": img_size,
            "num_classes": num_classes,
            "n_train_images": n_train_images,
            "n_val_images": n_val_images,
            "n_test_images": n_test_images,
            "augmentations": str(apply_augmentations),
            "batch_size": batch_size,
            "epochs": epochs,
        },
    )
    wandb.log(
        {
            "training_time_mins": training_time_mins,
            "test_accuracy": round(test_accuracy[1], 4),
            "test2_accuracy": round(test2_accuracy[1], 4),
            "precision": round(precision, 4),
            "recall": round(recall, 4),
            "f1": round(f1, 4),
        }
    )
    artifact = wandb.Artifact("classification_results", type="evaluation")
    artifact.add_file(classification_report_path)
    artifact.add_file(confusion_matrix_path)
    wandb.log_artifact(artifact)
    wandb.finish()